# Phase 1: Layer A synthetic population

Builds a population of agents from 2021 Census tables for NSW using iterative proportional fitting (IPF), then checks the agents against those tables.

The census tables are produced by `python -m src.population.census_gcp` (see the README). If they are missing, this notebook falls back to the fake test tables and labels every chart as fake.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.population.schema import CORE_ATTRIBUTES
from src.population.tables import load_constraints
from src.population.ipf import ipf
from src.population.sample import sample_agents
from src.population import validate

REAL = ROOT / 'data' / 'processed' / 'tables'
TABLES_DIR = REAL if any(REAL.glob('*.csv')) else ROOT / 'data' / 'sample_fake'
SOURCE_LABEL = 'FAKE test tables (not census)' if 'fake' in TABLES_DIR.name else 'Census 2021'
N_AGENTS = 2000
print('Using tables from:', TABLES_DIR.relative_to(ROOT), '|', SOURCE_LABEL)

## 1. Load the tables
Each table becomes a constraint the population must match.

In [ ]:
constraints = load_constraints(TABLES_DIR)
for name, c in constraints.items():
    print(f'{name:26s} {" x ".join(c.dims):32s} {c.target.size} cells')

## 2. Fit the joint distribution with IPF
No single table covers all six attributes. IPF finds one six-way distribution that agrees with all of them.

Four tables come from the Community Profile and are broken down by location, age and sex. The fifth, `age_income_work`, comes from TableBuilder and ties income to work status at each age. Without it, income and work would be treated as unrelated within each location, age and sex group.

**Known limitation.** Living arrangement is still linked to income and work only through location, age and sex.

In [ ]:
dim_names = list(CORE_ATTRIBUTES)
dim_sizes = [len(v) for v in CORE_ATTRIBUTES.values()]
fit = ipf(dim_names, dim_sizes, list(constraints.values()))
print('converged:', fit.converged, '| iterations:', fit.iterations, '| max error:', f'{fit.max_error:.2e}')
print('cells in joint table:', fit.table.size, '| non-empty:', int((fit.table > 0).sum()))

## 3. Sample agents

In [ ]:
agents = sample_agents(fit.table, dim_names, n=N_AGENTS, seed=42)
agents.head(10)

## 4. Validate
Gaps are in percentage points. With 2,000 agents, gaps of one to two points are normal sampling noise; they shrink as the number of agents grows.

In [ ]:
validate.compare_tables(agents, constraints)

In [ ]:
marginals = validate.compare_marginals(agents, constraints)
print('agents in impossible combinations:', validate.structural_zero_violations(agents, constraints))
marginals

In [ ]:
from IPython.display import Image
path = validate.plot_marginals(marginals, ROOT / 'outputs' / 'validation' / 'marginals.png', source_label=SOURCE_LABEL)
Image(str(path))

## 5. A look at the relationships
Full-time workers should rarely have nil income, and people outside the labour force should mostly have low incomes.

In [ ]:
pd.crosstab(agents.age, agents.work, normalize='index').round(2)

In [ ]:
pd.crosstab(agents.work, agents.income, normalize='index').round(2)

## 6. Save

In [ ]:
out = ROOT / 'data' / 'processed' / 'agents_layer_a.csv'
agents.to_csv(out, index=False)
print(out.relative_to(ROOT))